# Session 2: COP, the Carnot limit and sizing a fridge
**A. Real-world problem.** A kitchen fridge holds 4 C inside while the kitchen is 25 C.
Heat leaks in through the walls all day. How much electric power must the compressor
use, and why does the same fridge use more power in a hot summer kitchen? Is there a
best possible fridge?

Educational model: ideal temperatures, steady state, one "fraction of Carnot" number
for all real losses. Not a design tool for real appliances.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
ZERO_C = 273.15   # 0 degrees Celsius in kelvin

## Step 1: kelvin, COP and the best possible COP
Temperature ratios must use kelvin: T(K) = T(C) + 273.15.

    COP = Q_cold / W                       (heat removed per unit of work)
    COP_Carnot = T_c / (T_h - T_c)         (no fridge can beat this)

`T_c` is the cold temperature and `T_h` the warm one, both in kelvin. A real fridge reaches
only a fraction `eta2` of the Carnot value: COP = eta2 x COP_Carnot.

In [ ]:
def kelvin(t_celsius):
    return t_celsius + ZERO_C


def cop_carnot(t_cold_c, t_hot_c):
    """Best possible COP between a cold space and a warm room (temperatures in Celsius)."""
    tc, th = kelvin(t_cold_c), kelvin(t_hot_c)
    if th <= tc:
        raise ValueError("the room must be warmer than the cold space")
    return tc / (th - tc)


def cop_real(t_cold_c, t_hot_c, eta2):
    """Real COP as a fraction eta2 (0 to 1) of the Carnot COP."""
    if not 0 < eta2 <= 1:
        raise ValueError("eta2 must be above 0 and at most 1")
    return eta2 * cop_carnot(t_cold_c, t_hot_c)

## Step 2: size the compressor for a heat leak
Steady state: heat removed Q_cold (per second) equals the leak. Then

    W = Q_cold / COP        (electric power, W)
    Q_hot = Q_cold + W      (heat given to the kitchen, W)

In [ ]:
def electric_power(leak_w, t_cold_c, t_hot_c, eta2):
    return leak_w / cop_real(t_cold_c, t_hot_c, eta2)


def room_heating(leak_w, t_cold_c, t_hot_c, eta2):
    """Q_hot = Q_cold + W, in watts."""
    return leak_w + electric_power(leak_w, t_cold_c, t_hot_c, eta2)

## Step 3: the design problem (inverse)
Given a power limit, how hot may the kitchen be? Solve W = Q_cold (T_h - T_c) / (eta2 T_c)
for the warm temperature:

    T_h = T_c + eta2 x T_c x W / Q_cold

In [ ]:
def max_room_temp_c(leak_w, t_cold_c, eta2, power_limit_w):
    """Warmest room (Celsius) for which the compressor power stays at or below the limit."""
    tc = kelvin(t_cold_c)
    return tc + eta2 * tc * power_limit_w / leak_w - ZERO_C

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    c = cop_carnot(4, 25)
    cr = cop_real(4, 25, 0.25)
    w = electric_power(60, 4, 25, 0.25)
    print(f"Carnot COP 4 C / 25 C = {c:.2f};  real COP (eta2 = 0.25) = {cr:.2f}")
    print(f"leak 60 W -> compressor {w:.1f} W, heat to kitchen {room_heating(60, 4, 25, 0.25):.1f} W")
    print(f"warmest room for 15 W limit (leak 60 W, eta2 0.30): {max_room_temp_c(60, 4, 0.30, 15):.1f} C")

    room = np.linspace(10, 45, 141)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for t_in, col in ((-18, "tab:purple"), (4, "tab:blue"), (10, "tab:green")):
        mask = room > t_in + 1
        ax.plot(room[mask], [cop_carnot(t_in, r) for r in room[mask]], color=col, label=f"inside {t_in} C")
    ax.plot(room, [cop_real(4, r, 0.25) for r in room], "k--", label="real fridge, 4 C, eta2 = 0.25")
    ax.set_ylim(0, 25)
    ax.set_xlabel("room temperature (C)")
    ax.set_ylabel("COP (no unit)")
    ax.set_title("Carnot COP falls as the room gets hotter")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "cop_vs_room.png", dpi=120)

    leak = np.linspace(10, 120, 56)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for th, col in ((20, "tab:green"), (25, "tab:blue"), (32, "tab:red")):
        ax2.plot(leak, [electric_power(q, 4, th, 0.25) for q in leak], color=col, label=f"room {th} C")
    ax2.axhline(15, color="gray", ls=":", label="15 W limit")
    ax2.set_xlabel("heat leak Q_cold (W)")
    ax2.set_ylabel("compressor power W (W)")
    ax2.set_title("Compressor power for a 4 C fridge, eta2 = 0.25")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "power_vs_leak.png", dpi=120)

## Where this stops
The Carnot limit uses the room and inside temperatures. In a real fridge the refrigerant must
be colder than the inside and hotter than the room, and that is why eta2 is small.
Session 3 builds a cycle and measures eta2 instead of assuming it.

## Coding tasks
1. **Easy.** Compute with `cop_carnot` the Carnot COP for a freezer at -18 C in a 25 C kitchen and compare it with the 4 C fridge. Predict which is larger first.
2. **Stretch.** Write `eta2_needed(leak_w, t_cold_c, t_hot_c, power_limit_w)` that returns the least eta2 meeting a power limit, and test it with 60 W, 4 C, 32 C and 15 W (you should get about 0.404).
3. **Explain.** Explain why a fridge placed next to an oven needs more electricity, using the formula for COP_Carnot.

## Assessment
- **A. Conceptual.** Why can no fridge have a COP above the Carnot value? And why does a freezer at -18 C have a lower Carnot COP than a fridge at 4 C in the same kitchen?
- **B. Numerical.** A fridge holds 5 C in a 30 C kitchen with a 40 W heat leak and eta2 = 0.30. Find the Carnot COP, the real COP, the compressor power and Q_hot.
- **C. Graph and simulation interpretation.** On the COP graph for a 4 C inside, estimate by what fraction the Carnot COP falls when the kitchen goes from 25 C to 35 C. Confirm with the formula.
- **D. Design and optimisation challenge.** Design: the heat leak is 60 W, the inside is 4 C and the compressor power must not exceed 15 W. Find the warmest kitchen allowed if eta2 = 0.30. Then, for a 32 C kitchen, find the least eta2 that meets the limit. State the trade-off.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    c = L.cop_carnot(4, 25)
    checks.append((f"worked example: Carnot COP between 4 C and 25 C is 277.15/21 = 13.20 (got {c:.2f})", abs(c - 277.15 / 21) < 1e-9))
    cr = L.cop_real(4, 25, 0.25)
    checks.append((f"worked example: real COP with eta2 = 0.25 is 3.30 (got {cr:.2f})", abs(cr - 3.2994) < 1e-3))
    w = L.electric_power(60, 4, 25, 0.25)
    checks.append((f"worked example: leak 60 W needs W = 18.2 W (got {w:.1f})", abs(w - 18.18) < 0.01))
    qh = L.room_heating(60, 4, 25, 0.25)
    checks.append(("energy balance: Q_hot = Q_cold + W", abs(qh - (60 + w)) < 1e-12))
    checks.append(("Q_hot is 78.2 W in the worked example", abs(qh - 78.19) < 0.01))
    checks.append(("kelvin(0) = 273.15 and kelvin(-273.15) = 0", abs(L.kelvin(0) - 273.15) < 1e-12 and abs(L.kelvin(-273.15)) < 1e-12))
    checks.append(("hotter room means a lower COP", L.cop_carnot(4, 35) < L.cop_carnot(4, 25) < L.cop_carnot(4, 15)))
    checks.append(("colder inside means a lower COP", L.cop_carnot(-18, 25) < L.cop_carnot(4, 25)))
    checks.append(("real COP never exceeds Carnot", all(L.cop_real(4, t, e) <= L.cop_carnot(4, t) + 1e-12 for t in (10, 25, 40) for e in (0.1, 0.5, 1.0))))
    # design problem: round trip of the inverse formula
    th = L.max_room_temp_c(60, 4, 0.30, 15)
    checks.append((f"design: warmest room for 15 W is 24.8 C (got {th:.1f})", abs(th - 24.786) < 0.01))
    checks.append(("design: round trip, power at that room temperature is exactly 15 W", abs(L.electric_power(60, 4, th, 0.30) - 15.0) < 1e-9))
    checks.append(("design: eta2 = 0.30 at 32 C fails the 15 W limit, eta2 = 0.45 passes",
                   L.electric_power(60, 4, 32, 0.30) > 15 and L.electric_power(60, 4, 32, 0.45) < 15))
    for bad, label in ((lambda: L.cop_carnot(25, 25), "equal temperatures"), (lambda: L.cop_carnot(30, 25), "room colder than inside"),
                       (lambda: L.cop_real(4, 25, 0.0), "eta2 = 0")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    checks.append(("edge case: room just above the inside gives a very large but finite COP", np.isfinite(L.cop_carnot(4, 4.001)) and L.cop_carnot(4, 4.001) > 1e4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
